# Lesson 2.5 — Selecting Built-in Tools: Read, Write, Edit, Bash, Grep, Glob

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Select Grep for content search and Glob for path-pattern matching
- Use Read/Write for full-file ops and Edit for targeted unique-match edits
- Fall back to Read + Write when Edit cannot find a unique anchor
- Build codebase understanding incrementally rather than reading everything upfront

**You will produce:** A built-in tool decision table plus an incremental Grep->Read tracing workflow.

## Concepts

The built-in tools each have a sweet spot:

- **Grep** - search file *contents* for patterns (function names, error messages, imports).
- **Glob** - match file *paths* by name/extension patterns (e.g. `**/*.test.tsx`).
- **Read / Write** - full-file operations.
- **Edit** - targeted modification via **unique** text matching. When the anchor text is not unique, Edit fails - fall back to **Read + Write** for a reliable change.

**Incremental exploration beats reading everything:** start with Grep to find entry points, then Read to follow imports and trace flows. To trace a function across wrapper modules, first identify all exported names, then Grep each name across the codebase.

> **Reference:** [Anthropic - Claude Code built-in tools](https://docs.anthropic.com/en/docs/claude-code)

In [ ]:
import os, pathlib, re
SRC = pathlib.Path('demo_src'); SRC.mkdir(exist_ok=True)
(SRC / 'orders.py').write_text('def create_order():\n    return 1\n'
                               'def cancel_order():\n    return 0\n', encoding='utf-8')
(SRC / 'orders.test.py').write_text('def test_create():\n    assert True\n', encoding='utf-8')
print('created', [p.name for p in SRC.iterdir()])

## Code-Along: Step 1 — Grep vs Glob

Grep searches contents; Glob matches paths. We emulate both with the stdlib.

In [ ]:
def grep(pattern, root='demo_src'):
    hits = []
    for p in pathlib.Path(root).rglob('*.py'):
        for i, line in enumerate(p.read_text(encoding='utf-8').splitlines(), 1):
            if re.search(pattern, line):
                hits.append((p.name, i, line.strip()))
    return hits

def glob(pattern, root='demo_src'):
    return [str(p) for p in pathlib.Path(root).rglob(pattern)]

print('grep def _order:', grep(r'def \w+_order'))
print('glob *.test.py:', glob('*.test.py'))

## Code-Along: Step 2 — Edit's unique-match requirement and the fallback

Edit needs a unique anchor. If the text appears twice, use Read + Write instead.

In [ ]:
def edit(path, old, new):
    text = pathlib.Path(path).read_text(encoding='utf-8')
    if text.count(old) != 1:
        raise ValueError(f"Edit failed: {text.count(old)} matches for {old!r}; "
                         "fall back to Read + Write")
    pathlib.Path(path).write_text(text.replace(old, new), encoding='utf-8')
    return 'edited'

try:
    edit('demo_src/orders.py', 'return', 'return  # changed')  # 'return' not unique
except ValueError as e:
    print('Expected:', e)
# Fallback: Read the whole file, modify precisely, Write it back.
p = pathlib.Path('demo_src/orders.py')
src = p.read_text(encoding='utf-8').replace('return 1', 'return 100')
p.write_text(src, encoding='utf-8')
print('fallback write OK:', 'return 100' in p.read_text(encoding='utf-8'))

## Code-Along: Step 3 — Incremental tracing: Grep entry points, then Read

Find callers first, then follow only the relevant files instead of reading all of them.

In [ ]:
entry_points = grep(r'def create_order')
print('entry points:', entry_points)
# Then Read just those files to follow imports/flows (incremental, context-efficient).
for name, _, _ in entry_points:
    print('would Read:', name)

## Exercise

Build a `select_tool(task)` helper returning the right built-in for: 'find all callers of refund()', 'list every **/*.test.tsx file', 'change one unique config line', and 'the anchor text appears 3 times'. The last must return 'Read+Write'. Add one assertion per case.

In [ ]:
# TODO: implement select_tool(task) -> 'Grep' | 'Glob' | 'Edit' | 'Read+Write'
# TODO: assert select_tool('find all callers of refund()') == 'Grep'
# TODO: assert select_tool('anchor appears 3 times') == 'Read+Write'

## Key Takeaways

- Grep searches contents; Glob matches file paths/extensions.
- Read/Write for full files; Edit for targeted edits needing a UNIQUE anchor.
- When Edit's anchor isn't unique, fall back to Read + Write.
- Explore incrementally: Grep entry points, then Read to follow flows - don't read everything.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.